In [4]:
import pandas as pd
from pathlib import Path

In [6]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

vector_path = PROJECT_ROOT / "data" / "processed" / "vector_retrieval_results.csv"
graph_path = PROJECT_ROOT / "data" / "processed" / "graph_retrieval_results.csv"

vector_results = pd.read_csv(vector_path)
graph_results = pd.read_csv(graph_path)

vector_results.head(), graph_results.head()

(  query_artist  rank retrieved_artist  similarity_score
 0  Soundgarden     1        Pearl Jam          0.740760
 1  Soundgarden     2          Nirvana          0.711962
 2  Soundgarden     3             Blur          0.684405
 3      Nirvana     1        Pearl Jam          0.754655
 4      Nirvana     2      Soundgarden          0.711962,
   query_artist  rank       retrieved_artist  graph_score evidence_type  \
 0         Blur     1              Pearl Jam            7    shared_tag   
 1         Blur     2  The Smashing Pumpkins            5    shared_tag   
 2         Blur     3                Nirvana            4    shared_tag   
 3     Deftones     1            Soundgarden            2    shared_tag   
 4     Deftones     2                Nirvana            2    shared_tag   
 
                                             evidence  
 0  ['alternative rock', 'grunge', 'rock', 'art ro...  
 1  ['alternative rock', 'grunge', 'rock', 'shoega...  
 2  ['alternative rock', 'grunge', 'n

In [7]:
graph_scores = graph_results.pivot_table(
    index=["query_artist", "retrieved_artist"],
    columns="evidence_type",
    values="graph_score",
    aggfunc="sum",
    fill_value=0
).reset_index()

graph_scores.columns.name = None

graph_scores.head()

,query_artist,retrieved_artist,shared_member,shared_tag
0,Blur,Nirvana,0,4
1,Blur,Pearl Jam,0,7
2,Blur,The Smashing Pumpkins,0,5
3,Deftones,Nirvana,0,2
4,Deftones,Soundgarden,0,2


In [8]:
for col in ["shared_member", "shared_tag"]:
    if col not in graph_scores.columns:
        graph_scores[col] = 0

graph_scores.head()

,query_artist,retrieved_artist,shared_member,shared_tag
0,Blur,Nirvana,0,4
1,Blur,Pearl Jam,0,7
2,Blur,The Smashing Pumpkins,0,5
3,Deftones,Nirvana,0,2
4,Deftones,Soundgarden,0,2


In [10]:
hybrid_candidates = vector_results.merge(
    graph_scores,
    on=["query_artist", "retrieved_artist"],
    how="left"
)

hybrid_candidates[["shared_member", "shared_tag"]] = hybrid_candidates[
    ["shared_member", "shared_tag"]
].fillna(0)

hybrid_candidates.head()

,query_artist,rank,retrieved_artist,similarity_score,shared_member,shared_tag
0,Soundgarden,1,Pearl Jam,0.740760,1.0,4.0
1,Soundgarden,2,Nirvana,0.711962,1.0,6.0
2,Soundgarden,3,Blur,0.684405,0.0,0.0
3,Nirvana,1,Pearl Jam,0.754655,0.0,7.0
4,Nirvana,2,Soundgarden,0.711962,1.0,6.0


In [11]:
hybrid_candidates["shared_tag_norm"] = hybrid_candidates.groupby("query_artist")["shared_tag"].transform(
    lambda x: x / x.max() if x.max() > 0 else 0
)

hybrid_candidates["graph_bonus"] = (
    (hybrid_candidates["shared_member"] > 0).astype(float) * 0.20
    + hybrid_candidates["shared_tag_norm"] * 0.10
)

hybrid_candidates["hybrid_score"] = (
    hybrid_candidates["similarity_score"]
    + hybrid_candidates["graph_bonus"]
)

hybrid_candidates.head()

,query_artist,rank,retrieved_artist,similarity_score,shared_member,shared_tag,shared_tag_norm,graph_bonus,hybrid_score
0,Soundgarden,1,Pearl Jam,0.740760,1.0,4.0,0.666667,0.266667,1.007427
1,Soundgarden,2,Nirvana,0.711962,1.0,6.0,1.000000,0.300000,1.011961
2,Soundgarden,3,Blur,0.684405,0.0,0.0,0.000000,0.000000,0.684405
3,Nirvana,1,Pearl Jam,0.754655,0.0,7.0,1.000000,0.100000,0.854655
4,Nirvana,2,Soundgarden,0.711962,1.0,6.0,0.857143,0.285714,0.997676


In [12]:
hybrid_results = hybrid_candidates.sort_values(
    ["query_artist", "hybrid_score"],
    ascending=[True, False]
).copy()

hybrid_results["hybrid_rank"] = hybrid_results.groupby("query_artist").cumcount() + 1

hybrid_results = hybrid_results[
    [
        "query_artist",
        "hybrid_rank",
        "retrieved_artist",
        "similarity_score",
        "shared_member",
        "shared_tag",
        "shared_tag_norm",
        "graph_bonus",
        "hybrid_score"
    ]
]

hybrid_results

,query_artist,hybrid_rank,retrieved_artist,similarity_score,shared_member,shared_tag,shared_tag_norm,graph_bonus,hybrid_score
15,Blur,1,Pearl Jam,0.760920,0.0,7.0,1.000000,0.100000,0.860920
16,Blur,2,The Smashing Pumpkins,0.696780,0.0,5.0,0.714286,0.071429,0.768209
17,Blur,3,Deftones,0.691868,0.0,0.0,0.000000,0.000000,0.691868
7,Deftones,1,Soundgarden,0.681798,0.0,2.0,1.000000,0.100000,0.781798
6,Deftones,2,Blur,0.691868,0.0,0.0,0.000000,0.000000,0.691868
8,Deftones,3,Pearl Jam,0.653498,0.0,0.0,0.000000,0.000000,0.653498
4,Nirvana,1,Soundgarden,0.711962,1.0,6.0,0.857143,0.285714,0.997676
3,Nirvana,2,Pearl Jam,0.754655,0.0,7.0,1.000000,0.100000,0.854655
5,Nirvana,3,Blur,0.675207,0.0,4.0,0.571429,0.057143,0.732350
11,Pearl Jam,1,Soundgarden,0.740760,1.0,4.0,0.571429,0.257143,0.997903


In [15]:
hybrid_results[hybrid_results["query_artist"] == "Pearl Jam"]

,query_artist,hybrid_rank,retrieved_artist,similarity_score,shared_member,shared_tag,shared_tag_norm,graph_bonus,hybrid_score
11,Pearl Jam,1,Soundgarden,0.740760,1.0,4.0,0.571429,0.257143,0.997903
9,Pearl Jam,2,Blur,0.760920,0.0,7.0,1.000000,0.100000,0.860920
10,Pearl Jam,3,Nirvana,0.754655,0.0,7.0,1.000000,0.100000,0.854655


In [14]:
hybrid_results[hybrid_results["query_artist"] == "Nirvana"]

,query_artist,hybrid_rank,retrieved_artist,similarity_score,shared_member,shared_tag,shared_tag_norm,graph_bonus,hybrid_score
4,Nirvana,1,Soundgarden,0.711962,1.0,6.0,0.857143,0.285714,0.997676
3,Nirvana,2,Pearl Jam,0.754655,0.0,7.0,1.000000,0.100000,0.854655
5,Nirvana,3,Blur,0.675207,0.0,4.0,0.571429,0.057143,0.732350


In [16]:
out_path = PROJECT_ROOT / "data" / "processed" / "hybrid_retrieval_results.csv"

hybrid_results.to_csv(out_path, index=False)

out_path

WindowsPath('c:/uni/seriousism/reminisGraph/data/processed/hybrid_retrieval_results.csv')

In [17]:
pd.read_csv(out_path).head()

,query_artist,hybrid_rank,retrieved_artist,similarity_score,shared_member,shared_tag,shared_tag_norm,graph_bonus,hybrid_score
0,Blur,1,Pearl Jam,0.760920,0.0,7.0,1.000000,0.100000,0.860920
1,Blur,2,The Smashing Pumpkins,0.696780,0.0,5.0,0.714286,0.071429,0.768209
2,Blur,3,Deftones,0.691868,0.0,0.0,0.000000,0.000000,0.691868
3,Deftones,1,Soundgarden,0.681798,0.0,2.0,1.000000,0.100000,0.781798
4,Deftones,2,Blur,0.691868,0.0,0.0,0.000000,0.000000,0.691868


#### Hybrid Scoring Design

This hybrid retriever combines vector similarity with graph-based evidence.

The vector score comes from cosine similarity between artist embeddings. This captures text and tag-based similarity.

The graph bonus comes from two graph evidence types:

1. **Shared members**, which represent historical or relational connections between artists.
2. **Shared tags**, which represent metadata or genre-style overlap.

Shared-member evidence receives a stronger bonus because it is more graph-specific and less likely to be captured by the vector baseline. Shared-tag evidence receives a smaller bonus because tags are already included in the vector embedding text.

The current hybrid score is a simple interpretable baseline, not a final optimized ranking formula.

#### Phase 9B Conclusion

This notebook builds a simple hybrid retrieval baseline by combining vector similarity with graph-based evidence.

The hybrid retriever starts from the vector retrieval results, then adds graph-based bonuses for artist pairs that are also connected in the Neo4j graph.

The hybrid score uses three signals:

1. **Vector similarity**, which captures album/tag text similarity.
2. **Shared-member evidence**, which captures historical or relational artist connections.
3. **Shared-tag evidence**, which captures metadata overlap in the graph.

This approach helps balance two types of discovery. Vector retrieval is useful for finding artists with similar text or genre descriptions, while graph retrieval can reveal explicit relationship evidence such as shared band members.

For example, if an artist pair has strong graph evidence through a shared member, the hybrid retriever can raise that pair in the ranking even if it was not the top vector result.

The current scoring formula is intentionally simple and interpretable. It is not meant to be a final production ranking model. Instead, it provides a baseline for understanding how vector and graph signals can be combined.

The output is saved as `hybrid_retrieval_results.csv`.

The next step is to evaluate vector, graph, and hybrid retrieval results side by side.